# `feats.py` — Real-Data Benchmark Notebook

Benchmarks the three `feats.py` utilities against **six real phonetic datasets** from HuggingFace,
all covering Ibero-Romance languages and dialects that live in `ipa-mappings`.

| Dataset | Language | Rows | Key columns |
|---|---|---|---|
| `TigreGotico/galician_g2p` | Galician (`gl`) | 10k | `sentence`, `phonemes` |
| `TigreGotico/mirandese_g2p` | Mirandese (`mwl`) | 219 | `dialect`, `word`, `ipa` |
| `TigreGotico/portuguese_phonetic_lexicon` | Portuguese (`pt`) | 100k+ | `word`, `ipa` |
| `TigreGotico/barranquenho-ipa-dict-synthetic` | Barranquenho (`ext-PT-x-barrancos`) | 319 | `barranquenho_orthography`, `ipa_transcription` |
| `TigreGotico/ArquivoDialetalCLUP_ipa` | PT dialects | small | allophone-rich |
| `fdemelo/ipa-childes-split` | 31 languages | 12.5M | `lang`, `sentence`, `ipa_espeak` |

### What this measures
1. **Tokenizer accuracy** — `PhonetokTokenizer` output vs gold IPA (phoneme error rate)
2. **LM phonotactics** — `build_ngram_lm` + `perplexity` on real word/sentence phoneme sequences
3. **Feature space** — `phoneme_embeddings` with inventories extracted from the gold data
4. **Cross-dataset perplexity matrix** — do Galician LM phonotactics transfer to Mirandese? to Barranquenho?
5. **Allophone analysis** — CLUP dialect data allophones vs package allophone maps
6. **Cross-lingual CHILDES** — multi-language perplexity and feature overlap

In [6]:
# ── Dependencies ──────────────────────────────────────────────────────────────
%pip install datasets==2.19.0        # HuggingFace datasets loader
%pip install numpy==1.26.4           # array ops
%pip install pandas==2.2.2           # data wrangling
%pip install scikit-learn==1.4.2     # edit distance, PCA, cosine sim
%pip install umap-learn==0.5.6       # UMAP projection
%pip install matplotlib==3.8.4       # plots
%pip install seaborn==0.13.2         # statistical plots
%pip install scipy==1.13.0           # hierarchical clustering
%pip install tqdm==4.66.2            # progress bars
%pip install psutil==5.9.8           # memory monitoring


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached pandas-2.2.2.tar.gz (4.4 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... \^C
anceled

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.
  Using cached scikit-learn-1.4.2.tar.gz (7.8 MB)
  Installing build dependencies ... error
  error: subprocess-exited-with-error
  
  × pip subprocess to install build dependencies did not run succe

In [7]:
import os, sys, gc, math, re
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import psutil
#from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────

OUTPUT_DIR    = os.getenv("OUTPUT_DIR", "./outputs")
NGRAM_N       = int(os.getenv("NGRAM_N", "3"))       # n-gram order for LMs
BEAM_WIDTH    = int(os.getenv("BEAM_WIDTH", "1"))    # tokenizer beam width
MAX_ROWS      = int(os.getenv("MAX_ROWS", "3000"))   # cap per dataset (speed)
CHILDES_LANGS = os.getenv("CHILDES_LANGS",           # CHILDES language codes to include
                           "pt-PT,pt-BR,es,ca,gl,fr,de,it,nl")
HF_TOKEN      = os.getenv("HF_TOKEN", None)          # optional, for gated datasets

os.makedirs(OUTPUT_DIR, exist_ok=True)


def mem():
    gc.collect()
    m = psutil.virtual_memory()
    print(f"🧠 RAM: {m.used/1e9:.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")

def savefig(name):
    path = f"{OUTPUT_DIR}/{name}"
    plt.savefig(path, dpi=140, bbox_inches="tight")
    print(f"  💾 {path}")
    return path

print(f"✅ Config | n={NGRAM_N}-gram | beam={BEAM_WIDTH} | max_rows={MAX_ROWS}")

✅ Config | n=3-gram | beam=1 | max_rows=3000


In [8]:
# ── Import package + feats.py ─────────────────────────────────────────────────
import orthography2ipa
from orthography2ipa.phonetok import PhonetokTokenizer
from orthography2ipa.feats import phoneme_embeddings, build_ngram_lm, perplexity

print(f"✅ orthography2ipa loaded — {len(orthography2ipa.available_codes())} codes")
print(f"✅ feats.py imported")

✅ orthography2ipa loaded — 223 codes
✅ feats.py imported


---
## 1. Load Datasets

In [10]:
from datasets import load_dataset

# ── Shared loader helper ──────────────────────────────────────────────────────
def load_hf(repo, split="train", max_rows=MAX_ROWS, token=HF_TOKEN, **kwargs):
    """Load a HuggingFace dataset, cap rows, return as DataFrame."""
    ds = load_dataset(repo, split=split, token=token, **kwargs)
    if max_rows and len(ds) > max_rows:
        ds = ds.select(range(max_rows))
    return ds.to_pandas()

print("Loading datasets…")

# ── 1. Galician G2P (sentence-level) ─────────────────────────────────────────
gl_df = load_hf("TigreGotico/galician_g2p")
# Normalise: split sentence words/phones into individual (word, ipa) pairs
gl_pairs = []
for _, row in gl_df.iterrows():
    words = str(row["sentence"]).split()
    phones = str(row["phonemes"]).split()
    if len(words) == len(phones):
        gl_pairs += list(zip(words, phones))
gl_words_df = pd.DataFrame(gl_pairs, columns=["word", "ipa"]).drop_duplicates("word")
print(f"  gl  — {len(gl_df)} sentences → {len(gl_words_df)} unique words")

# ── 2. Mirandese G2P (word-level, has dialect column) ─────────────────────────
mwl_df = load_hf("TigreGotico/mirandese_g2p", max_rows=None)  # tiny dataset, load all
mwl_df = mwl_df.rename(columns={"ipa": "ipa"})
print(f"  mwl — {len(mwl_df)} rows | dialects: {sorted(mwl_df['dialect'].unique())}")

# ── 3. Portuguese Phonetic Lexicon ────────────────────────────────────────────
pt_df = load_hf("TigreGotico/portuguese_phonetic_lexicon")
# Detect word/ipa column names
pt_word_col = [c for c in pt_df.columns if "word" in c.lower() or "ortho" in c.lower()][0]
pt_ipa_col  = [c for c in pt_df.columns if "ipa" in c.lower() or "phon" in c.lower()][0]
pt_df = pt_df[[pt_word_col, pt_ipa_col]].rename(columns={pt_word_col: "word", pt_ipa_col: "ipa"})
pt_df = pt_df.dropna().drop_duplicates("word")
print(f"  pt  — {len(pt_df)} unique words")

# ── 4. Barranquenho ───────────────────────────────────────────────────────────
barr_df = load_hf("TigreGotico/barranquenho-ipa-dict-synthetic", max_rows=None)
barr_df = barr_df.rename(columns={"barranquenho_orthography": "word",
                                   "ipa_transcription": "ipa"})[["word", "ipa"]]
barr_df = barr_df.dropna().drop_duplicates("word")
print(f"  barr — {len(barr_df)} unique words")

# ── 5. ArquivoDialetalCLUP ────────────────────────────────────────────────────
clup_df = load_hf("TigreGotico/ArquivoDialetalCLUP_ipa", max_rows=None)
print(f"  CLUP — {len(clup_df)} rows | columns: {list(clup_df.columns)}")

# ── 6. CHILDES (large — sample only) ─────────────────────────────────────────
childes_df = load_hf("fdemelo/ipa-childes-split", max_rows=MAX_ROWS)
childes_df = childes_df[["lang", "sentence", "ipa_espeak"]].dropna()
print(f"  CHILDES — {len(childes_df)} rows | langs: {sorted(childes_df['lang'].unique())}")

mem()

Loading datasets…


TypeError: Pickler._batch_setitems() takes 2 positional arguments but 3 were given

In [ ]:
# ── Canonical dataset registry ────────────────────────────────────────────────
# Maps dataset name → (DataFrame with 'word'/'sentence' + 'ipa', ipa-mappings code)

DATASETS = {
    "Galician (gl)":        (gl_words_df,  "gl",                "word", "ipa"),
    "Mirandese (mwl)": (mwl_df[["word","ipa"]], "mwl",          "word", "ipa"),
    "Portuguese (pt)":      (pt_df,         "pt",               "word", "ipa"),
    "Barranquenho":         (barr_df,        "ext-PT-x-barrancos", "word", "ipa"),
}

# CHILDES per-language slice
childes_lang_map = {
    "pt-PT": "pt", "pt-BR": "pt-BR", "es": "es",
    "ca": "ca", "gl": "gl", "fr": "fr",
    "de": "de", "it": "it", "nl": "nl",
}
wanted = set(CHILDES_LANGS.split(","))
for childes_code, pkg_code in childes_lang_map.items():
    if childes_code not in wanted:
        continue
    sub = childes_df[childes_df["lang"] == childes_code][["sentence","ipa_espeak"]].rename(
        columns={"sentence": "word", "ipa_espeak": "ipa"}).dropna().head(MAX_ROWS)
    if len(sub) > 0:
        DATASETS[f"CHILDES-{childes_code}"] = (sub, pkg_code, "word", "ipa")

print("✅ Dataset registry:")
for name, (df, code, wc, ic) in DATASETS.items():
    print(f"   {name:35s}  code={code:22s}  rows={len(df)}")

---
## 2. EDA — Dataset Overview

In [ ]:
# ── Per-dataset IPA token inventories ─────────────────────────────────────────
def ipa_tokens(ipa_str: str) -> list[str]:
    """Split an IPA string into individual phone tokens.
    Handles diacritics (ˈˌ̃̈̂) and multi-char phones greedily."""
    # Simple: split on whitespace if spaces present, otherwise char-by-char
    s = str(ipa_str).strip()
    if " " in s:
        return [t for t in s.split() if t]
    # Char-level, but keep combining diacritics attached
    import unicodedata
    tokens, buf = [], ""
    for ch in s:
        cat = unicodedata.category(ch)
        if buf and cat.startswith("M"):  # combining mark → attach to prev
            buf += ch
        elif buf and ch in "ːˑ˥˦˧˨˩":
            buf += ch
        else:
            if buf: tokens.append(buf)
            buf = ch
    if buf: tokens.append(buf)
    return [t for t in tokens if t not in ("ˈ","ˌ","/","[","]","|")]

# Compute stats per dataset
stats = []
ds_inventories = {}
for name, (df, code, wc, ic) in DATASETS.items():
    all_tokens = [t for ipa in df[ic].dropna() for t in ipa_tokens(ipa)]
    freq = Counter(all_tokens)
    ds_inventories[name] = freq
    stats.append({
        "dataset": name, "lang_code": code,
        "n_entries": len(df),
        "unique_ipa_tokens": len(freq),
        "total_ipa_tokens": sum(freq.values()),
        "avg_ipa_len": np.mean([len(ipa_tokens(i)) for i in df[ic].dropna()[:500]]),
    })
    
stats_df = pd.DataFrame(stats)
print(stats_df.to_string(index=False))

In [ ]:
# ── Figure 1: Dataset summary — entries, vocab size, avg IPA length ───────────
core_ds = [n for n in DATASETS if not n.startswith("CHILDES")]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

core_stats = stats_df[stats_df["dataset"].isin(core_ds)]

# Bar 1: entry count
axes[0].barh(core_stats["dataset"], core_stats["n_entries"], color="steelblue")
axes[0].set_xlabel("Number of entries")
axes[0].set_title("Dataset sizes")

# Bar 2: unique IPA token count
axes[1].barh(core_stats["dataset"], core_stats["unique_ipa_tokens"], color="darkorange")
axes[1].set_xlabel("Unique IPA tokens")
axes[1].set_title("Phoneme inventory size (data)")

# Bar 3: avg IPA string length
axes[2].barh(core_stats["dataset"], core_stats["avg_ipa_len"], color="seagreen")
axes[2].set_xlabel("Avg IPA tokens per entry")
axes[2].set_title("Average phoneme sequence length")

plt.suptitle("Dataset Overview", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("01_dataset_overview.png")
plt.show()

In [ ]:
# ── Figure 2: Top-20 IPA tokens per dataset ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, name in zip(axes.flat, core_ds):
    freq = ds_inventories[name]
    top = freq.most_common(20)
    tokens, counts = zip(*top)
    ax.barh(list(tokens)[::-1], list(counts)[::-1], color="steelblue")
    ax.set_title(f"{name}")
    ax.set_xlabel("Count")
    ax.tick_params(axis='y', labelsize=9)

plt.suptitle("Top-20 IPA Tokens per Dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("02_top_ipa_tokens.png")
plt.show()

In [ ]:
# ── Figure 3: IPA inventory overlap heatmap (Jaccard) ─────────────────────────
# How much of each dataset's IPA vocabulary is shared with others?

ds_names = list(core_ds)
inventories = {n: set(ds_inventories[n].keys()) for n in ds_names}
n = len(ds_names)
jaccard_mat = np.zeros((n, n))

for i, a in enumerate(ds_names):
    for j, b in enumerate(ds_names):
        inter = len(inventories[a] & inventories[b])
        union = len(inventories[a] | inventories[b])
        jaccard_mat[i, j] = inter / union if union else 0.0

fig, ax = plt.subplots(figsize=(7, 6))
im = sns.heatmap(jaccard_mat, annot=True, fmt=".2f",
                 xticklabels=ds_names, yticklabels=ds_names,
                 cmap="YlGnBu", ax=ax, linewidths=0.5,
                 cbar_kws={"label": "Jaccard similarity"})
ax.set_title("IPA Inventory Overlap (Jaccard)\nbetween datasets")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
savefig("03_inventory_jaccard.png")
plt.show()

---
## 3. Tokenizer Accuracy — Package vs Gold IPA

In [ ]:
# ── Phoneme Error Rate (PER) using Levenshtein edit distance ─────────────────

def levenshtein(a: list, b: list) -> int:
    """Edit distance between two token lists."""
    m, n = len(a), len(b)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            prev, dp[j] = dp[j], prev if a[i-1] == b[j-1] else 1 + min(prev, dp[j], dp[j-1])
    return dp[n]

def per(pred_ipa: str, gold_ipa: str) -> float:
    """Phoneme Error Rate: edit distance / len(gold)."""
    pred_t = ipa_tokens(pred_ipa)
    gold_t = ipa_tokens(gold_ipa)
    if not gold_t: return 1.0
    return levenshtein(pred_t, gold_t) / len(gold_t)

def benchmark_tokenizer(df, lang_code, word_col, ipa_col, n=500):
    """Compare PhonetokTokenizer output vs gold IPA for a sample of rows."""
    try:
        spec = orthography2ipa.get(lang_code)
        tok  = PhonetokTokenizer(spec)
    except Exception as e:
        print(f"  ⚠️  {lang_code}: {e}")
        return None

    sample = df[[word_col, ipa_col]].dropna().head(n)
    pers, exact = [], []
    errors = []  # (word, pred, gold, per)
    for _, row in sample.iterrows():
        word = str(row[word_col]).strip().lower()
        gold = str(row[ipa_col]).strip()
        try:
            paths = tok.ipa_beam(word, beam_width=BEAM_WIDTH)
            pred  = paths[0].ipa if paths else ""
        except Exception:
            pred = ""
        p = per(pred, gold)
        pers.append(p)
        exact.append(int(p == 0.0))
        if p > 0.3:
            errors.append({"word": word, "pred": pred, "gold": gold, "per": round(p, 3)})

    return {
        "mean_per":    np.mean(pers),
        "median_per":  np.median(pers),
        "exact_match": np.mean(exact),
        "per_dist":    pers,
        "worst":       sorted(errors, key=lambda x: -x["per"])[:10],
    }

print("Benchmarking tokenizer accuracy (this may take a moment)...")
tok_results = {}
for name, (df, code, wc, ic) in DATASETS.items():
    if name.startswith("CHILDES"): continue  # do CHILDES separately
    r = benchmark_tokenizer(df, code, wc, ic)
    if r:
        tok_results[name] = r
        print(f"  {name:35s}  PER={r['mean_per']:.3f}  exact={r['exact_match']:.2%}")

mem()

In [ ]:
# ── Figure 4: PER summary + distribution ─────────────────────────────────────
fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1, 2])

# Left: mean PER bar chart
ax0 = fig.add_subplot(gs[0])
names  = list(tok_results.keys())
mean_p = [tok_results[n]["mean_per"] for n in names]
exact  = [tok_results[n]["exact_match"] for n in names]
y = range(len(names))
bars = ax0.barh(y, mean_p, color="tomato", label="Mean PER")
ax0.barh(y, exact, color="seagreen", alpha=0.6, label="Exact match rate")
ax0.set_yticks(y)
ax0.set_yticklabels(names, fontsize=9)
ax0.set_xlabel("Rate")
ax0.set_title("Tokenizer Accuracy\nvs Gold IPA")
ax0.set_xlim(0, 1)
ax0.legend(fontsize=8)
ax0.axvline(0.1, color="gray", linestyle=":", alpha=0.5)

# Right: PER distribution violin
ax1 = fig.add_subplot(gs[1])
all_per_data  = [tok_results[n]["per_dist"] for n in names]
vp = ax1.violinplot(all_per_data, positions=range(len(names)),
                    showmedians=True, showextrema=True)
for body in vp["bodies"]:
    body.set_alpha(0.6)
ax1.set_xticks(range(len(names)))
ax1.set_xticklabels(names, rotation=25, ha="right", fontsize=8)
ax1.set_ylabel("PER per entry")
ax1.set_title("PER Distribution per Dataset")
ax1.axhline(0.0, color="green", linestyle="--", alpha=0.4, label="Perfect")
ax1.legend(fontsize=8)

plt.suptitle("PhonetokTokenizer: Phoneme Error Rate vs Gold IPA",
             fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("04_per_benchmark.png")
plt.show()

In [ ]:
# ── Figure 5: Worst-case errors per dataset ───────────────────────────────────
for name, r in tok_results.items():
    if not r["worst"]: continue
    print(f"\n── {name} — Highest-PER examples ──")
    err_df = pd.DataFrame(r["worst"])
    print(err_df[["word","pred","gold","per"]].to_string(index=False))

---
## 4. N-Gram LM — Phonotactic Benchmarks on Gold IPA

In [ ]:
# ── Build LMs from gold IPA sequences ─────────────────────────────────────────
# Instead of synthetic words, we feed the GOLD IPA phone sequences directly
# to build_ngram_lm (which calls the tokenizer internally).
# For the gold-IPA evaluation we also compute perplexity of the LM on the
# *original orthographic words* — this measures how well the package's
# phonotactic model matches the data distribution.

# Split each dataset 80/20 train/test
lm_results = {}   # name -> {lm, train_ppl, test_ppl, lang_code}

for name, (df, code, wc, ic) in DATASETS.items():
    if name.startswith("CHILDES"): continue
    try:
        orthography2ipa.get(code)  # ensure code is valid
    except Exception:
        print(f"  ⚠️  Skipping {name} — code {code} not in registry")
        continue
    words = df[wc].dropna().str.lower().tolist()
    split = int(len(words) * 0.8)
    train_w, test_w = words[:split], words[split:]
    lm = build_ngram_lm(train_w, code, n=NGRAM_N)
    tr_ppl = perplexity(lm, train_w[:200], code, n=NGRAM_N)
    te_ppl = perplexity(lm, test_w[:200],  code, n=NGRAM_N)
    lm_results[name] = {
        "lm": lm, "lang_code": code,
        "train_words": train_w, "test_words": test_w,
        "train_ppl": tr_ppl, "test_ppl": te_ppl,
        "n_contexts": len(lm),
    }
    print(f"  {name:35s}  train_ppl={tr_ppl:.1f}  test_ppl={te_ppl:.1f}  ctx={len(lm):,}")

mem()

In [ ]:
# ── Figure 6: Train vs Test PPL + N-gram order sweep ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: train vs test PPL bar chart
ax = axes[0]
lm_names = list(lm_results.keys())
tr_ppls = [lm_results[n]["train_ppl"] for n in lm_names]
te_ppls = [lm_results[n]["test_ppl"]  for n in lm_names]
x = np.arange(len(lm_names))
w = 0.35
ax.bar(x - w/2, tr_ppls, w, label="Train PPL", color="steelblue")
ax.bar(x + w/2, te_ppls, w, label="Test PPL",  color="tomato")
ax.set_xticks(x); ax.set_xticklabels(lm_names, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("Perplexity"); ax.set_title(f"{NGRAM_N}-gram LM Perplexity per Dataset")
ax.legend()

# Right: n-gram sweep on Galician (largest dataset)
ax2 = axes[1]
sweep_name = "Galician (gl)" if "Galician (gl)" in lm_results else lm_names[0]
sweep_r = lm_results[sweep_name]
ns, ppls, ctxs = [], [], []
for n_val in range(1, 6):
    lm_n = build_ngram_lm(sweep_r["train_words"], sweep_r["lang_code"], n=n_val)
    p = perplexity(lm_n, sweep_r["test_words"][:200], sweep_r["lang_code"], n=n_val)
    ns.append(n_val); ppls.append(p); ctxs.append(len(lm_n))

color1, color2 = "steelblue", "darkorange"
ax2.plot(ns, ppls, marker="o", color=color1, label="Test PPL")
ax2.set_xlabel("n-gram order"); ax2.set_ylabel("PPL", color=color1)
ax2.tick_params(axis="y", labelcolor=color1)
ax2b = ax2.twinx()
ax2b.plot(ns, ctxs, marker="s", color=color2, linestyle="--", label="# contexts")
ax2b.set_ylabel("# distinct contexts", color=color2)
ax2b.tick_params(axis="y", labelcolor=color2)
ax2.set_title(f"N-gram Order Sweep — {sweep_name}")
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=8)

plt.suptitle("Phonotactic LM Benchmarks", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("05_lm_perplexity.png")
plt.show()

In [ ]:
# ── Figure 7: Per-word PPL distribution and word length correlation ───────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pick the two most interesting datasets to compare
pair = lm_names[:2]
all_word_ppls = {}  # name -> list of (word_len, ppl)

for name in pair:
    r = lm_results[name]
    rows = []
    for w in r["test_words"][:300]:
        p = perplexity(r["lm"], [w], r["lang_code"], n=NGRAM_N)
        if p != float("inf"):
            rows.append({"word": w, "ppl": p, "len": len(w)})
    all_word_ppls[name] = pd.DataFrame(rows)

# Left: side-by-side histogram
for name in pair:
    d = all_word_ppls[name]["ppl"].clip(upper=all_word_ppls[name]["ppl"].quantile(0.98))
    axes[0].hist(d, bins=30, alpha=0.6, label=name, density=True)
axes[0].set_xlabel("Per-word perplexity"); axes[0].set_ylabel("Density")
axes[0].set_title("Per-Word PPL Distribution")
axes[0].legend(fontsize=8)

# Right: word length vs PPL scatter
colors = ["steelblue", "darkorange"]
for i, name in enumerate(pair):
    d = all_word_ppls[name]
    axes[1].scatter(d["len"], d["ppl"].clip(upper=d["ppl"].quantile(0.95)),
                    alpha=0.4, s=15, color=colors[i], label=name)
    # Trend line
    if len(d) > 3:
        z = np.polyfit(d["len"], d["ppl"].clip(upper=d["ppl"].quantile(0.95)), 1)
        xs = np.linspace(d["len"].min(), d["len"].max(), 50)
        axes[1].plot(xs, np.polyval(z, xs), color=colors[i], linewidth=2)

axes[1].set_xlabel("Word length (chars)"); axes[1].set_ylabel("Per-word PPL")
axes[1].set_title("Word Length vs Perplexity")
axes[1].legend(fontsize=8)

plt.suptitle("Per-Word Phonotactic Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("06_word_ppl_analysis.png")
plt.show()

---
## 5. Cross-Dataset Perplexity Matrix

In [ ]:
# ── Cross-perplexity: LM from dataset A scored on words from dataset B ────────
cross_names = lm_names  # only the core datasets
N_ds = len(cross_names)
ppl_mat = np.full((N_ds, N_ds), np.nan)

for i, lm_name in enumerate(cross_names):
    for j, word_name in enumerate(cross_names):
        lm_r    = lm_results[lm_name]
        word_r  = lm_results[word_name]
        # Use lm_name's LM but tokenize via lm_name's lang (the LM's phonology)
        # Test words come from word_name dataset
        p = perplexity(lm_r["lm"], word_r["test_words"][:150],
                       lm_r["lang_code"], n=NGRAM_N)
        ppl_mat[i, j] = p if p != float("inf") else np.nan

ppl_df = pd.DataFrame(ppl_mat, index=cross_names, columns=cross_names)
print("Cross-dataset perplexity (row=LM, col=test words):")
print(ppl_df.round(1).to_string())

In [ ]:
# ── Figure 8: Cross-perplexity heatmap + dendrogram ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: heatmap
sns.heatmap(ppl_df.round(1), annot=True, fmt=".1f",
            cmap="YlOrRd_r", ax=axes[0],
            linewidths=0.5, mask=np.isnan(ppl_mat),
            cbar_kws={"label": "PPL (lower = more phonotactically similar)"})
axes[0].set_title("Cross-Dataset Phonotactic Perplexity\n(row=LM, col=test words)")
axes[0].tick_params(axis="x", rotation=25)

# Right: dendogram from symmetric PPL distance
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

sym = (ppl_mat + ppl_mat.T) / 2
sym = np.nan_to_num(sym, nan=np.nanmax(sym[~np.isinf(sym)]) * 1.5 if not np.all(np.isnan(sym)) else 1.0)
np.fill_diagonal(sym, 0.0)
dist_norm = sym / (sym.max() + 1e-8)

if N_ds >= 3:
    Z = linkage(squareform(dist_norm, checks=False), method="average")
    dendrogram(Z, labels=cross_names, ax=axes[1], leaf_rotation=30)
    axes[1].set_title("Phonotactic Distance Dendrogram\n(from cross-PPL)")
    axes[1].set_ylabel("Distance")
else:
    axes[1].text(0.5, 0.5, "Need ≥3 datasets", ha="center", va="center")

plt.suptitle("Cross-Dataset Phonotactic Similarity", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("07_crosslingual_ppl.png")
plt.show()

---
## 6. `phoneme_embeddings()` — Feature Space vs Gold Data

In [ ]:
# ── Compare package inventory vs data inventory per language ──────────────────
# For each core dataset:
#   - Package inventory = phonemes in spec.graphemes (via phoneme_embeddings)
#   - Data inventory    = unique IPA tokens in the gold dataset

coverage_rows = []
for name, (df, code, wc, ic) in DATASETS.items():
    if name.startswith("CHILDES"): continue
    try:
        pkg_embeds = phoneme_embeddings(code)
    except Exception:
        continue
    pkg_inv  = set(pkg_embeds.keys())
    data_inv = inventories[name]
    
    in_both        = pkg_inv & data_inv
    pkg_only       = pkg_inv - data_inv
    data_only      = data_inv - pkg_inv
    coverage_rows.append({
        "dataset": name, "lang": code,
        "pkg_size":   len(pkg_inv),
        "data_size":  len(data_inv),
        "shared":     len(in_both),
        "pkg_only":   len(pkg_only),
        "data_only":  len(data_only),
        "coverage":   len(in_both) / len(data_inv) if data_inv else 0,
        "missing_phones": sorted(data_only)[:10],
    })

cov_df = pd.DataFrame(coverage_rows)
print(cov_df[["dataset","pkg_size","data_size","shared","data_only","coverage"]].to_string(index=False))
print("\nPhones in gold data not in package (top-10 per dataset):")
for _, row in cov_df.iterrows():
    print(f"  {row['dataset']:35s}  {row['missing_phones']}")

In [ ]:
# ── Figure 9: Coverage stacked bar ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
x = np.arange(len(cov_df))
ax.bar(x, cov_df["shared"],   label="Shared",     color="seagreen")
ax.bar(x, cov_df["pkg_only"],  bottom=cov_df["shared"], label="Pkg only",  color="steelblue")
ax.bar(x, cov_df["data_only"], bottom=cov_df["shared"]+cov_df["pkg_only"],
       label="Data only (gap)", color="tomato")
ax.set_xticks(x); ax.set_xticklabels(cov_df["dataset"], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("# IPA tokens")
ax.set_title("Package vs Data Phoneme Inventory")
ax.legend(fontsize=8)

ax2 = axes[1]
ax2.barh(cov_df["dataset"], cov_df["coverage"] * 100, color="seagreen")
ax2.set_xlabel("Coverage (%)")
ax2.set_title("Package IPA Coverage\n(% of data inventory covered by package)")
ax2.axvline(80, color="orange", linestyle="--", alpha=0.7, label="80% threshold")
ax2.axvline(100, color="green", linestyle="--", alpha=0.7, label="100%")
ax2.legend(fontsize=8)
for i, (_, row) in enumerate(cov_df.iterrows()):
    ax2.text(row["coverage"]*100 + 0.5, i, f"{row['coverage']:.0%}", va="center", fontsize=8)

plt.suptitle("IPA Inventory Coverage Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("08_inventory_coverage.png")
plt.show()

In [ ]:
# ── Figure 10: Feature space — package phones vs data phones ──────────────────
# For a chosen language, overlay package embeddings (circles) with
# data-only phones (triangles = gaps) in the same 2D projection.

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

focus = "Galician (gl)"
focus_code = DATASETS[focus][1]

pkg_emb = phoneme_embeddings(focus_code)
data_inv_set = inventories[focus]

# Phones we can embed from the package
pkg_phones  = sorted(pkg_emb.keys())
pkg_vecs    = np.stack([pkg_emb[p] for p in pkg_phones])

# Data-only phones: try to embed them too via vectorize_phones
from orthography2ipa.distance import vectorize_phones as vp_fn
data_only_phones, data_only_vecs = [], []
for p in sorted(data_inv_set - set(pkg_phones)):
    try:
        v = vp_fn(p)
        data_only_vecs.append(np.array([0.5 if x is None else float(x) for x in v]))
        data_only_phones.append(p)
    except Exception:
        pass

all_vecs = np.vstack([pkg_vecs] + ([np.stack(data_only_vecs)] if data_only_vecs else []))
coords = PCA(n_components=2).fit_transform(all_vecs)

fig, ax = plt.subplots(figsize=(11, 8))
n_pkg = len(pkg_phones)
ax.scatter(coords[:n_pkg, 0], coords[:n_pkg, 1],
           c="steelblue", s=80, marker="o", label="Package phonemes", zorder=3)
if data_only_phones:
    ax.scatter(coords[n_pkg:, 0], coords[n_pkg:, 1],
               c="tomato", s=100, marker="^", label="In data only (gap)", zorder=4)
    for i, p in enumerate(data_only_phones):
        ax.annotate(p, (coords[n_pkg+i, 0], coords[n_pkg+i, 1]),
                    xytext=(4, 4), textcoords="offset points", fontsize=9,
                    color="tomato", fontweight="bold")
for i, p in enumerate(pkg_phones):
    ax.annotate(p, (coords[i, 0], coords[i, 1]),
                xytext=(3, 3), textcoords="offset points", fontsize=8, color="#1f4e79")
ax.set_title(f"Phoneme Feature Space — {focus}\n"
             f"(PCA of 21-dim articulatory vectors; ▲ = phones in data not in package)")
ax.set_xlabel("PC-1"); ax.set_ylabel("PC-2")
ax.legend(fontsize=9)
plt.tight_layout()
savefig("09_feature_space_coverage.png")
plt.show()

---
## 7. Mirandese Dialect Analysis

In [ ]:
# ── Figure 11: Mirandese dialects — IPA inventory overlap + per-dialect PPL ──
dialects = mwl_df["dialect"].unique()
print(f"Mirandese dialects: {sorted(dialects)}")

spec_mwl = orthography2ipa.get("mwl")
tok_mwl  = PhonetokTokenizer(spec_mwl)

dialect_data = {}
for d in sorted(dialects):
    sub = mwl_df[mwl_df["dialect"] == d]
    inv = Counter(t for ipa in sub["ipa"].dropna() for t in ipa_tokens(ipa))
    # PER for this dialect
    pers_d = []
    for _, row in sub.iterrows():
        try:
            pred = tok_mwl.ipa_beam(str(row["word"]).lower(), beam_width=BEAM_WIDTH)
            p = per(pred[0].ipa if pred else "", str(row["ipa"]))
        except Exception:
            p = 1.0
        pers_d.append(p)
    dialect_data[d] = {"inv": inv, "per": pers_d, "n": len(sub)}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: PER violin per dialect
ax = axes[0]
data_to_plot = [dialect_data[d]["per"] for d in sorted(dialects)]
labels = sorted(dialects)
vp = ax.violinplot(data_to_plot, showmedians=True)
for body in vp["bodies"]: body.set_alpha(0.6)
ax.set_xticks(range(1, len(labels)+1)); ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("PER"); ax.set_title("Mirandese: PER by dialect")

# Right: inventory overlap between dialects
d_list = sorted(dialects)
jaccard_d = np.zeros((len(d_list), len(d_list)))
for i, a in enumerate(d_list):
    for j, b in enumerate(d_list):
        sa = set(dialect_data[a]["inv"].keys())
        sb = set(dialect_data[b]["inv"].keys())
        inter = len(sa & sb); union = len(sa | sb)
        jaccard_d[i, j] = inter / union if union else 0
sns.heatmap(jaccard_d, annot=True, fmt=".2f",
            xticklabels=d_list, yticklabels=d_list,
            cmap="Blues", ax=axes[1])
axes[1].set_title("Mirandese: Dialect Inventory Overlap (Jaccard)")

plt.suptitle("Mirandese Dialect Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("10_mirandese_dialects.png")
plt.show()

---
## 8. ArquivoDialetalCLUP — Allophone Analysis

In [ ]:
# ── Explore CLUP schema ───────────────────────────────────────────────────────
print("CLUP columns:", list(clup_df.columns))
print(clup_df.head(5).to_string())

In [ ]:
# ── Figure 12: CLUP allophone inventory vs package allophone map ──────────────
# Identify the IPA/allophone column(s)
ipa_cols = [c for c in clup_df.columns if "ipa" in c.lower() or "phon" in c.lower() or "alloph" in c.lower()]
dialect_col = next((c for c in clup_df.columns if "dial" in c.lower() or "region" in c.lower() or "place" in c.lower()), None)
word_col_clup = next((c for c in clup_df.columns if "word" in c.lower() or "lemma" in c.lower() or "ortho" in c.lower()), None)

print(f"IPA cols: {ipa_cols}")
print(f"Dialect col: {dialect_col}")
print(f"Word col: {word_col_clup}")

# Extract all IPA tokens from all IPA columns
clup_all_tokens = Counter()
for col in ipa_cols:
    for val in clup_df[col].dropna():
        for t in ipa_tokens(str(val)):
            clup_all_tokens[t] += 1

# Compare with pt package allophones
pt_spec = orthography2ipa.get("pt")
pkg_allophones = set(t for al in pt_spec.allophones.values() for t in al)
pkg_phonemes   = set(phoneme_embeddings("pt").keys())

clup_inv = set(clup_all_tokens.keys())
print(f"\nCLUP unique tokens:   {len(clup_inv)}")
print(f"Package phonemes:     {len(pkg_phonemes)}")
print(f"Package allophones:   {len(pkg_allophones)}")
print(f"In CLUP not in pkg:   {sorted(clup_inv - pkg_phonemes - pkg_allophones)[:15]}")
print(f"In pkg not in CLUP:   {sorted(pkg_phonemes - clup_inv)[:15]}")

In [ ]:
# ── Figure 12: CLUP token frequency + dialect breakdown ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: top token frequencies, color-coded by membership
ax = axes[0]
top_clup = clup_all_tokens.most_common(30)
colors_clup = []
for tok, _ in top_clup:
    if tok in pkg_phonemes: colors_clup.append("steelblue")
    elif tok in pkg_allophones: colors_clup.append("darkorange")
    else: colors_clup.append("tomato")

toks, cnts = zip(*top_clup)
ax.barh(list(toks)[::-1], list(cnts)[::-1], color=colors_clup[::-1])
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="steelblue",  label="In pkg phonemes"),
    Patch(color="darkorange", label="In pkg allophones"),
    Patch(color="tomato",     label="Not in package"),
], fontsize=8)
ax.set_title("CLUP IPA Token Frequency\n(vs Portuguese package inventory)")
ax.set_xlabel("Count")
ax.tick_params(axis="y", labelsize=9)

# Right: dialect inventory sizes (if dialect column present)
ax2 = axes[1]
if dialect_col:
    dial_invs = {}
    for d, grp in clup_df.groupby(dialect_col):
        tokens = set()
        for col in ipa_cols:
            for v in grp[col].dropna():
                tokens.update(ipa_tokens(str(v)))
        dial_invs[str(d)] = len(tokens)
    sorted_dials = sorted(dial_invs.items(), key=lambda x: -x[1])
    ax2.barh([d for d,_ in sorted_dials], [n for _,n in sorted_dials], color="seagreen")
    ax2.axvline(len(pkg_phonemes), color="steelblue", linestyle="--", label=f"pkg phonemes ({len(pkg_phonemes)})")
    ax2.axvline(len(pkg_allophones), color="darkorange", linestyle="--", label=f"pkg allophones ({len(pkg_allophones)})")
    ax2.set_xlabel("Unique IPA tokens")
    ax2.set_title("CLUP: IPA Inventory Size by Dialect")
    ax2.legend(fontsize=8)
else:
    ax2.text(0.5, 0.5, "No dialect column detected", ha="center", va="center", transform=ax2.transAxes)

plt.suptitle("CLUP Dialect Archive — Allophone Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
savefig("11_clup_allophones.png")
plt.show()

---
## 9. CHILDES Multi-Language Benchmark

In [ ]:
# ── CHILDES: phoneme inventory per language + feature space ───────────────────
childes_ds = {n: v for n, v in DATASETS.items() if n.startswith("CHILDES")}
print(f"CHILDES subsets loaded: {list(childes_ds.keys())}")

childes_stats = []
childes_embeds = {}  # lang_code -> phoneme_embeddings dict

for name, (df, code, wc, ic) in childes_ds.items():
    all_tok = [t for ipa in df[ic].dropna() for t in ipa_tokens(str(ipa))]
    freq = Counter(all_tok)
    try:
        e = phoneme_embeddings(code)
        childes_embeds[name] = e
    except Exception:
        pass
    childes_stats.append({
        "dataset": name, "lang_code": code,
        "n_sentences": len(df),
        "unique_phones": len(freq),
        "total_phones":  sum(freq.values()),
    })

ch_stats_df = pd.DataFrame(childes_stats)
print(ch_stats_df.to_string(index=False))

In [ ]:
# ── Figure 13: CHILDES cross-lingual feature space ────────────────────────────
# Overlay phoneme feature embeddings for all CHILDES languages

all_vecs_ch, all_phones_ch, all_langs_ch = [], [], []
for name, e in childes_embeds.items():
    for p, v in e.items():
        all_vecs_ch.append(v)
        all_phones_ch.append(p)
        all_langs_ch.append(name)

if len(all_vecs_ch) >= 5:
    from sklearn.decomposition import PCA
    try:
        import umap
        proj = umap.UMAP(n_components=2, random_state=42,
                         n_neighbors=min(10, len(all_vecs_ch)-1)).fit_transform(np.stack(all_vecs_ch))
        proj_lbl = "UMAP"
    except ImportError:
        proj = PCA(n_components=2).fit_transform(np.stack(all_vecs_ch))
        proj_lbl = "PCA"

    unique_chl = sorted(set(all_langs_ch))
    pal_ch = plt.cm.tab20(np.linspace(0, 1, len(unique_chl)))

    fig, ax = plt.subplots(figsize=(13, 8))
    for i, lang in enumerate(unique_chl):
        mask = [k for k, l in enumerate(all_langs_ch) if l == lang]
        ax.scatter(proj[mask, 0], proj[mask, 1],
                   c=[pal_ch[i]], label=lang, s=25, alpha=0.65)

    # Annotate cross-lingual anchor phones (appear in ≥60% of langs)
    phone_lang_count = Counter(all_phones_ch)
    anchors = {p for p, c in phone_lang_count.items() if c >= len(unique_chl) * 0.6}
    for k, (p, lang) in enumerate(zip(all_phones_ch, all_langs_ch)):
        if p in anchors and lang == unique_chl[0]:
            ax.annotate(p, (proj[k, 0], proj[k, 1]), fontsize=8,
                        color="black", fontweight="bold")

    ax.legend(fontsize=7, markerscale=2, ncol=2, loc="best")
    ax.set_title(f"CHILDES: Cross-Lingual Phoneme Feature Space ({proj_lbl})\n"
                 f"Bold = phones in ≥60% of languages")
    ax.set_xlabel(f"{proj_lbl}-1"); ax.set_ylabel(f"{proj_lbl}-2")
    plt.tight_layout()
    savefig("12_childes_feature_space.png")
    plt.show()

In [ ]:
# ── Figure 14: CHILDES cross-lingual PPL matrix ───────────────────────────────
# Build LMs for each CHILDES language and compute cross-PPL

# Build CHILDES LMs from sentence IPA tokens (treat each sentence as a "word")
childes_lms = {}
for name, (df, code, wc, ic) in childes_ds.items():
    try:
        orthography2ipa.get(code)
    except Exception:
        continue
    # Use sentences (space-separated, pass raw sentence text as word list)
    words = df[wc].dropna().str.lower().tolist()
    lm_ch = build_ngram_lm(words[:1000], code, n=NGRAM_N)
    test_ch = words[1000:1200]
    childes_lms[name] = {"lm": lm_ch, "code": code, "test": test_ch}

ch_names = list(childes_lms.keys())
N_ch = len(ch_names)

if N_ch >= 2:
    ch_ppl_mat = np.full((N_ch, N_ch), np.nan)
    for i, lm_name in enumerate(ch_names):
        for j, word_name in enumerate(ch_names):
            lr = childes_lms[lm_name]
            wr = childes_lms[word_name]
            p = perplexity(lr["lm"], wr["test"][:100], lr["code"], n=NGRAM_N)
            ch_ppl_mat[i, j] = p if p != float("inf") else np.nan

    ch_ppl_df = pd.DataFrame(ch_ppl_mat, index=ch_names, columns=ch_names)
    
    fig, ax = plt.subplots(figsize=(max(7, N_ch), max(6, N_ch * 0.8)))
    sns.heatmap(ch_ppl_df.round(1), annot=True, fmt=".1f",
                cmap="YlOrRd_r", ax=ax, linewidths=0.4,
                mask=np.isnan(ch_ppl_mat),
                cbar_kws={"label": "Perplexity"})
    ax.set_title(f"CHILDES: Cross-Language Phonotactic PPL\n(row=LM, col=test words, {NGRAM_N}-gram)")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout()
    savefig("13_childes_cross_ppl.png")
    plt.show()

---
## 10. Feature Similarity vs Perplexity — Correlation

In [ ]:
# ── Figure 15: Does feature similarity predict cross-PPL? ─────────────────────
# For every pair of (core) datasets, compute:
#   - avg cosine sim of shared phoneme feature vectors
#   - symmetric avg cross-PPL

from sklearn.metrics.pairwise import cosine_similarity as cos_sim

pair_data = []
for i, name_a in enumerate(cross_names):
    for j, name_b in enumerate(cross_names):
        if i >= j: continue
        if np.isnan(ppl_mat[i, j]): continue
        try:
            ea = phoneme_embeddings(DATASETS[name_a][1])
            eb = phoneme_embeddings(DATASETS[name_b][1])
        except Exception:
            continue
        shared = set(ea) & set(eb)
        if len(shared) < 3: continue
        va = np.stack([ea[p] for p in shared])
        vb = np.stack([eb[p] for p in shared])
        avg_sim = float(np.mean([cos_sim(va[k:k+1], vb[k:k+1])[0,0] for k in range(len(shared))]))
        avg_ppl = (ppl_mat[i,j] + ppl_mat[j,i]) / 2
        pair_data.append({"pair": f"{name_a}↔{name_b}",
                          "feat_sim": avg_sim, "cross_ppl": avg_ppl})

if len(pair_data) >= 3:
    from scipy.stats import pearsonr, spearmanr
    pd_df = pd.DataFrame(pair_data)
    r_p, p_p = pearsonr(pd_df["feat_sim"], pd_df["cross_ppl"])
    r_s, p_s = spearmanr(pd_df["feat_sim"], pd_df["cross_ppl"])
    print(f"Pearson r={r_p:.3f} (p={p_p:.4f})  |  Spearman ρ={r_s:.3f} (p={p_s:.4f})")

    fig, ax = plt.subplots(figsize=(9, 6))
    ax.scatter(pd_df["feat_sim"], pd_df["cross_ppl"], s=70, color="steelblue", zorder=3)
    for _, row in pd_df.iterrows():
        ax.annotate(row["pair"], (row["feat_sim"], row["cross_ppl"]),
                    xytext=(4,4), textcoords="offset points", fontsize=7)
    z = np.polyfit(pd_df["feat_sim"], pd_df["cross_ppl"], 1)
    xs = np.linspace(pd_df["feat_sim"].min(), pd_df["feat_sim"].max(), 50)
    ax.plot(xs, np.polyval(z, xs), "r--", label=f"r={r_p:.2f}, ρ={r_s:.2f}")
    ax.set_xlabel("Avg articulatory feature similarity (shared phonemes)")
    ax.set_ylabel("Avg cross-PPL")
    ax.set_title("Feature Similarity vs Phonotactic Perplexity\n"
                 "(negative correlation = articulatorily similar → lower cross-PPL)")
    ax.legend(fontsize=9)
    plt.tight_layout()
    savefig("14_feat_sim_vs_ppl.png")
    plt.show()
else:
    print("⚠️  Not enough language pairs — add more COMPARE_LANGS")

---
## 11. Summary Report

In [ ]:
# ── Figure 16: Summary dashboard ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

# Panel A: PER summary
ax_a = fig.add_subplot(gs[0, 0])
per_data = [(n, tok_results[n]["mean_per"], tok_results[n]["exact_match"])
            for n in tok_results]
per_names, per_vals, exact_vals = zip(*per_data)
ax_a.bar(range(len(per_names)), per_vals, color="tomato", label="Mean PER", alpha=0.8)
ax_a.bar(range(len(per_names)), exact_vals, color="seagreen", label="Exact match", alpha=0.8)
ax_a.set_xticks(range(len(per_names)))
ax_a.set_xticklabels(per_names, rotation=30, ha="right", fontsize=7)
ax_a.set_title("A. Tokenizer PER", fontsize=10, fontweight="bold")
ax_a.legend(fontsize=7)

# Panel B: Test PPL by dataset
ax_b = fig.add_subplot(gs[0, 1])
ax_b.bar(range(len(lm_names)), te_ppls, color="steelblue")
ax_b.set_xticks(range(len(lm_names)))
ax_b.set_xticklabels(lm_names, rotation=30, ha="right", fontsize=7)
ax_b.set_title(f"B. {NGRAM_N}-gram LM Test PPL", fontsize=10, fontweight="bold")

# Panel C: Inventory coverage
ax_c = fig.add_subplot(gs[0, 2])
ax_c.barh(cov_df["dataset"], cov_df["coverage"] * 100, color="seagreen")
ax_c.axvline(80, color="orange", linestyle="--", alpha=0.7)
ax_c.set_xlabel("Coverage %")
ax_c.set_title("C. Package IPA Coverage", fontsize=10, fontweight="bold")

# Panel D: Cross-PPL heatmap (small)
ax_d = fig.add_subplot(gs[1, 0])
sns.heatmap(ppl_df.round(0), annot=True, fmt=".0f", cmap="YlOrRd_r",
            ax=ax_d, cbar=False, linewidths=0.4,
            xticklabels=[n.split("(")[0].strip()[:8] for n in ppl_df.columns],
            yticklabels=[n.split("(")[0].strip()[:8] for n in ppl_df.index],
            mask=np.isnan(ppl_mat))
ax_d.set_title("D. Cross-PPL Matrix", fontsize=10, fontweight="bold")
ax_d.tick_params(labelsize=7)

# Panel E: Inventory Jaccard
ax_e = fig.add_subplot(gs[1, 1])
sns.heatmap(jaccard_mat, annot=True, fmt=".2f", cmap="YlGnBu",
            ax=ax_e, cbar=False, linewidths=0.4,
            xticklabels=[n.split("(")[0].strip()[:8] for n in ds_names],
            yticklabels=[n.split("(")[0].strip()[:8] for n in ds_names])
ax_e.set_title("E. IPA Inventory Jaccard", fontsize=10, fontweight="bold")
ax_e.tick_params(labelsize=7)

# Panel F: Stats table
ax_f = fig.add_subplot(gs[1, 2])
ax_f.axis("off")
summary_rows_final = []
for name in lm_names:
    per_val = tok_results.get(name, {}).get("mean_per", float("nan"))
    te_ppl_ = lm_results[name]["test_ppl"]
    cov_val = cov_df[cov_df["dataset"]==name]["coverage"].values
    cov_val = float(cov_val[0]) if len(cov_val) else float("nan")
    summary_rows_final.append([name.split("(")[0].strip()[:15],
                                f"{per_val:.2f}", f"{te_ppl_:.1f}", f"{cov_val:.0%}"])
tbl = ax_f.table(
    cellText=summary_rows_final,
    colLabels=["Dataset", "PER↓", "PPL↓", "Cov↑"],
    loc="center", cellLoc="center"
)
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
tbl.scale(1, 1.4)
ax_f.set_title("F. Summary Table", fontsize=10, fontweight="bold")

plt.suptitle("feats.py Real-Data Benchmark — Summary Dashboard",
             fontsize=14, fontweight="bold", y=1.01)
savefig("00_summary_dashboard.png")
plt.show()

print("\n✅ All plots saved to", OUTPUT_DIR)
mem()